In [1]:
import pandas as pd
from pathlib import Path
BASE_DIR = Path.cwd().parent.parent
# print(BASE_DIR)
INPUT_CSV = BASE_DIR / r"museumschina.cn\data\data_local_image_20260311_181727.csv"
df = pd.read_csv(INPUT_CSV)

In [2]:
# 展示列名，大小
print(df.columns)
print(df.shape)

Index(['detail_url', 'main_image_url', 'title_full', 'museum_detail',
       'category_detail', 'era_detail', 'level', 'entry_year', 'material',
       'gallery_urls', 'local_main_image', 'local_gallery_images'],
      dtype='object')
(1003, 12)


In [3]:
# 1. 标记这一行是否包含有效的本地图片
# 假设空字符串或NaN都视为无效
mask_missing_local = df['local_main_image'].isna() | (df['local_main_image'] == '')

# 收集因缺少图片而删除的行
df_dropped_missing = df[mask_missing_local].copy()
df_dropped_missing['drop_reason'] = 'missing_local_image'

# 剩下的行，进行第二步检查
df_remaining = df[~mask_missing_local].copy()

# 2. 检查 main_image_url 重复情况
# 找出所有重复出现的 main_image_url
dup_mask = df_remaining.duplicated(subset=['main_image_url'], keep=False)

df_unique = df_remaining[~dup_mask] # 唯一出现的，直接保留
df_check_dups = df_remaining[dup_mask] # 重复出现的，需要逻辑判断

rows_to_drop_list = []
rows_to_keep_list = [df_unique]

# 获取用于比较的列（排除 detail_url）
compare_cols = [c for c in df.columns if c != 'detail_url']

# 按 main_image_url 分组处理
for main_url, group in df_check_dups.groupby('main_image_url'):
    # 检查除 detail_url 外的其他列是否完全一致
    #通过 drop_duplicates 看剩下几行
    deduped = group.drop_duplicates(subset=compare_cols)
    
    if len(deduped) == 1:
        # 情况1：完全重复（除 detail_url 外）
        # 保留其中任一行（这里取第一行），其余删除
        keep = group.iloc[[0]]
        drop = group.iloc[1:].copy()
        
        rows_to_keep_list.append(keep)
        if not drop.empty:
            drop['drop_reason'] = 'redundant_duplicate'
            rows_to_drop_list.append(drop)
    else:
        # 情况2：存在矛盾（除 detail_url 外仍有不同）
        # 全部删除
        drop = group.copy()
        drop['drop_reason'] = 'conflicting_duplicate'
        rows_to_drop_list.append(drop)

# 合并所有待删除的行
df_dropped_dups = pd.concat(rows_to_drop_list) if rows_to_drop_list else pd.DataFrame()
df_dropped = pd.concat([df_dropped_missing, df_dropped_dups], ignore_index=False)

# 合并所有保留的行
df_clean = pd.concat(rows_to_keep_list).sort_index()

# 结果展示
print(f"原始行数: {len(df)}")
print(f"待删除行数 (df_dropped): {len(df_dropped)}")
print(f"  - 缺少本地图片: {len(df_dropped_missing)}")
print(f"  - 重复/矛盾: {len(df_dropped_dups)}")
print(f"保留行数 (df_clean): {len(df_clean)}")

# 显示待删除数据的样例以便检查
df_dropped.head()

原始行数: 1003
待删除行数 (df_dropped): 17
  - 缺少本地图片: 14
  - 重复/矛盾: 3
保留行数 (df_clean): 986


,detail_url,main_image_url,title_full,museum_detail,category_detail,era_detail,level,entry_year,material,gallery_urls,local_main_image,local_gallery_images,drop_reason
30,https://www.museumschina.cn/collection/details...,http://www.museumschina.cn/img/yipu/da/42/06/0...,新石器时代附加堆纹灰陶罐,湖北省襄阳市襄州区数字博物馆,陶器,新石器时代,未定级,2004.0,单一/无机质/陶,http://www.museumschina.cn/img/yipu/da/42/06/0...,NaN,NaN,missing_local_image
128,https://www.museumschina.cn/collection/details...,http://www.museumschina.cn/img/waterMark/2019-...,新石器时代晚期红陶网坠,望江县博物馆,陶器,新石器时代,三级,1980.0,单一/无机质/陶,http://www.museumschina.cn/img/waterMark/2019-...,NaN,NaN,missing_local_image
132,https://www.museumschina.cn/collection/details...,http://www.museumschina.cn/img/waterMark/2019-...,新石器时代陶纺轮,太湖县文物管理所（博物馆）,陶器,新石器时代,三级,2012.0,单一/无机质/陶,http://www.museumschina.cn/img/waterMark/2019-...,NaN,NaN,missing_local_image
241,https://www.museumschina.cn/collection/details...,http://www.museumschina.cn/img/yipu/da/42/10/8...,新石器时代灰陶双腹豆,石首市博物馆,陶器,新石器时代,三级,1990.0,单一/无机质/陶,http://www.museumschina.cn/img/yipu/da/42/10/8...,NaN,NaN,missing_local_image
254,https://www.museumschina.cn/collection/details...,http://www.museumschina.cn/img/waterMark/2019-...,新石器时代灰黑陶纺轮,望江县博物馆,陶器,新石器时代,三级,1983.0,单一/无机质/陶,http://www.museumschina.cn/img/waterMark/2019-...,NaN,NaN,missing_local_image


In [4]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 986 entries, 0 to 1002
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   detail_url            986 non-null    object 
 1   main_image_url        986 non-null    object 
 2   title_full            986 non-null    object 
 3   museum_detail         986 non-null    object 
 4   category_detail       986 non-null    object 
 5   era_detail            986 non-null    object 
 6   level                 986 non-null    object 
 7   entry_year            820 non-null    float64
 8   material              986 non-null    object 
 9   gallery_urls          986 non-null    object 
 10  local_main_image      986 non-null    object 
 11  local_gallery_images  986 non-null    object 
dtypes: float64(1), object(11)
memory usage: 100.1+ KB


In [8]:
# 去掉df_clean中的网络url，只保留本地路径
cols = df_clean.columns.to_list()
new_cols = [cols[0]] + cols[2:-3] + cols[-2:]
print(cols,"\n",new_cols)
df_final = df_clean[new_cols]
df_final.info()

['detail_url', 'main_image_url', 'title_full', 'museum_detail', 'category_detail', 'era_detail', 'level', 'entry_year', 'material', 'gallery_urls', 'local_main_image', 'local_gallery_images'] 
 ['detail_url', 'title_full', 'museum_detail', 'category_detail', 'era_detail', 'level', 'entry_year', 'material', 'local_main_image', 'local_gallery_images']
<class 'pandas.core.frame.DataFrame'>
Index: 986 entries, 0 to 1002
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   detail_url            986 non-null    object 
 1   title_full            986 non-null    object 
 2   museum_detail         986 non-null    object 
 3   category_detail       986 non-null    object 
 4   era_detail            986 non-null    object 
 5   level                 986 non-null    object 
 6   entry_year            820 non-null    float64
 7   material              986 non-null    object 
 8   local_main_image      986 non-nu

In [9]:
# 改列名
df_final.rename(columns={
    "detail_url" : "url",
    "title_full" : "name",
    "museum_detail" : "museum",
    "category_detail" : "category",
    "era_detail" : "era",
    "local_main_image": "main_image_path",
    "local_gallery_images": "all_images_paths"
}, inplace=True)

C:\Users\NOVA\AppData\Local\Temp\ipykernel_26528\3571260750.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final.rename(columns={


In [10]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 986 entries, 0 to 1002
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   url               986 non-null    object 
 1   name              986 non-null    object 
 2   museum            986 non-null    object 
 3   category          986 non-null    object 
 4   era               986 non-null    object 
 5   level             986 non-null    object 
 6   entry_year        820 non-null    float64
 7   material          986 non-null    object 
 8   main_image_path   986 non-null    object 
 9   all_images_paths  986 non-null    object 
dtypes: float64(1), object(9)
memory usage: 84.7+ KB


In [11]:
# 将df_clean保存到新的CSV文件
from datetime import datetime
TIME_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_CSV = BASE_DIR / rf"museumschina.cn\data\data_final_{TIME_STAMP}.csv"
df_final.to_csv(OUTPUT_CSV, index=False)